In [1]:
# SQL Analysis — TVDE Lisbon Dataset
# Using SQLite to query the cleaned dataset with SQL

import sqlite3
import pandas as pd

# Load cleaned data
df = pd.read_csv("../data/processed/tvde_rides_clean.csv")
# Fix inconsistent platform naming
df["Platform"] = df["Platform"].str.strip().str.title()

# Fix category naming (same cleanup as main notebook)
df["Category"] = df["Category"].str.strip().str.title()

# Create SQLite database in memory
conn = sqlite3.connect(":memory:")

# Remove auxiliary validation columns if present
cols_to_drop = ["Duration_Calculated", "Duration_Diff"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Load dataframe into SQL table
df.to_sql("rides", conn, index=False, if_exists="replace")

print("Database created successfully.")
print(f"Table 'rides' loaded with {len(df)} rows.")
print(f"\nColumns available:")
for col in df.columns:
    print(f"  - {col}")

Database created successfully.
Table 'rides' loaded with 1545 rows.

Columns available:
  - Ride_ID
  - Platform
  - Category
  - Date
  - Start_Time
  - End_Time
  - Duration_Min
  - Origin_PostCode
  - Dest_PostCode
  - Distance_Km
  - Client_Fare_EUR
  - Driver_Earnings_EUR


In [2]:
# ── GROUP 1: OVERVIEW & VOLUME ──────────────────────────────────────

def run_query(query, conn):
    return pd.read_sql_query(query, conn)

# Q1.1 — General dataset overview
q1_1 = """
SELECT
    COUNT(*)                                    AS total_rides,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(DISTINCT Platform)                    AS platforms,
    ROUND(SUM(Distance_Km), 2)                  AS total_km,
    ROUND(SUM(Duration_Min) / 60.0, 2)          AS total_hours,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_gross_earnings,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings_per_ride,
    ROUND(MIN(Driver_Earnings_EUR), 2)          AS min_ride_earnings,
    ROUND(MAX(Driver_Earnings_EUR), 2)          AS max_ride_earnings
FROM rides;
"""
print("Q1.1 — Dataset Overview")
print("-" * 55)
print(run_query(q1_1, conn).to_string(index=False))

# Q1.2 — Rides and earnings by platform
q1_2 = """
SELECT
    Platform,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_earnings,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings_per_ride,
    ROUND(SUM(Distance_Km), 2)                  AS total_km,
    ROUND(AVG(Distance_Km), 2)                  AS avg_km_per_ride
FROM rides
GROUP BY Platform
ORDER BY total_rides DESC;
"""
print("\nQ1.2 — Rides and Earnings by Platform")
print("-" * 55)
print(run_query(q1_2, conn).to_string(index=False))

# Q1.3 — Rides by category
q1_3 = """
SELECT
    Category,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Distance_Km), 2)                  AS avg_km,
    ROUND(AVG(Driver_Earnings_EUR / Distance_Km), 2)
                                                AS avg_eur_per_km
FROM rides
GROUP BY Category
ORDER BY total_rides DESC;
"""
print("\nQ1.3 — Rides by Category")
print("-" * 55)
print(run_query(q1_3, conn).to_string(index=False))

Q1.1 — Dataset Overview
-------------------------------------------------------
 total_rides  days_worked  platforms  total_km  total_hours  total_gross_earnings  avg_earnings_per_ride  min_ride_earnings  max_ride_earnings
        1545           96          2  15213.94        434.1               9961.36                   6.45               2.89              23.75

Q1.2 — Rides and Earnings by Platform
-------------------------------------------------------
Platform  total_rides  pct_rides  total_earnings  avg_earnings_per_ride  total_km  avg_km_per_ride
    Uber          794       51.4         5196.68                   6.54   8030.65            10.11
    Bolt          751       48.6         4764.68                   6.34   7183.29             9.56

Q1.3 — Rides by Category
-------------------------------------------------------
       Category  total_rides  pct_rides  avg_earnings  avg_km  avg_eur_per_km
         Uber X          615       39.8          6.52   10.67            0.77
    

In [3]:
# ── GROUP 2: EARNINGS & PROFITABILITY ───────────────────────────────

# Q2.1 — Top 10 best earning days
q2_1 = """
SELECT
    Date,
    COUNT(*)                                    AS total_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS daily_gross,
    ROUND(SUM(Distance_Km), 2)                  AS daily_km,
    ROUND(SUM(Duration_Min) / 60.0, 2)          AS daily_hours,
    ROUND(SUM(Driver_Earnings_EUR) /
          (SUM(Duration_Min) / 60.0), 2)        AS eur_per_hour
FROM rides
GROUP BY Date
ORDER BY daily_gross DESC
LIMIT 10;
"""
print("Q2.1 — Top 10 Best Earning Days")
print("-" * 65)
print(run_query(q2_1, conn).to_string(index=False))

# Q2.2 — Daily earnings above and below target
q2_2 = """
SELECT
    CASE
        WHEN SUM(Driver_Earnings_EUR) >= 100 THEN 'Above target (≥100€)'
        WHEN SUM(Driver_Earnings_EUR) >= 80  THEN 'Close (80–100€)'
        ELSE 'Below (< 80€)'
    END                                         AS performance,
    COUNT(*)                                    AS days,
    ROUND(AVG(SUM(Driver_Earnings_EUR)), 2)     AS avg_earnings
FROM rides
GROUP BY Date
GROUP BY performance
ORDER BY avg_earnings DESC;
"""

# Simplified version for SQLite compatibility
q2_2 = """
SELECT
    performance,
    COUNT(*)                                    AS days,
    ROUND(AVG(daily_gross), 2)                  AS avg_earnings
FROM (
    SELECT
        Date,
        SUM(Driver_Earnings_EUR)                AS daily_gross,
        CASE
            WHEN SUM(Driver_Earnings_EUR) >= 100
                THEN 'Above target (100+ EUR)'
            WHEN SUM(Driver_Earnings_EUR) >= 80
                THEN 'Close (80-100 EUR)'
            ELSE 'Below (under 80 EUR)'
        END                                     AS performance
    FROM rides
    GROUP BY Date
)
GROUP BY performance
ORDER BY avg_earnings DESC;
"""
print("\nQ2.2 — Daily Performance vs Target")
print("-" * 65)
print(run_query(q2_2, conn).to_string(index=False))

# Q2.3 — Earnings efficiency by distance bracket
q2_3 = """
SELECT
    CASE
        WHEN Distance_Km < 3    THEN '0-3 km'
        WHEN Distance_Km < 5    THEN '3-5 km'
        WHEN Distance_Km < 8    THEN '5-8 km'
        WHEN Distance_Km < 12   THEN '8-12 km'
        WHEN Distance_Km < 18   THEN '12-18 km'
        WHEN Distance_Km < 25   THEN '18-25 km'
        ELSE '25+ km'
    END                                         AS distance_bracket,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM rides
GROUP BY distance_bracket
ORDER BY
    CASE distance_bracket
        WHEN '0-3 km'   THEN 1
        WHEN '3-5 km'   THEN 2
        WHEN '5-8 km'   THEN 3
        WHEN '8-12 km'  THEN 4
        WHEN '12-18 km' THEN 5
        WHEN '18-25 km' THEN 6
        ELSE 7
    END;
"""
print("\nQ2.3 — Earnings Efficiency by Distance Bracket")
print("-" * 65)
print(run_query(q2_3, conn).to_string(index=False))

Q2.1 — Top 10 Best Earning Days
-----------------------------------------------------------------
      Date  total_rides  daily_gross  daily_km  daily_hours  eur_per_hour
2026-06-03           19       155.80    182.42         5.85         26.63
2026-05-24           23       153.20    245.85         6.13         24.98
2026-05-29           20       144.93    151.45         4.45         32.57
2026-05-30           19       133.04    181.69         4.33         30.70
2026-03-08           22       130.69    187.41         5.00         26.14
2026-03-07           19       129.15    228.78         6.30         20.50
2026-04-07           23       128.83    203.46         5.52         23.35
2026-06-01           19       128.69    153.76         4.27         30.16
2026-05-22           13       128.37    185.55         4.67         27.51
2026-04-29           17       128.34    221.20         6.07         21.15

Q2.2 — Daily Performance vs Target
----------------------------------------------------

In [4]:
# ── GROUP 3: TEMPORAL PATTERNS ──────────────────────────────────────

# Q3.1 — Average earnings by day of week
q3_1 = """
SELECT
    CASE CAST(strftime('%w', Date) AS INTEGER)
        WHEN 0 THEN '7-Sunday'
        WHEN 1 THEN '1-Monday'
        WHEN 2 THEN '2-Tuesday'
        WHEN 3 THEN '3-Wednesday'
        WHEN 4 THEN '4-Thursday'
        WHEN 5 THEN '5-Friday'
        WHEN 6 THEN '6-Saturday'
    END                                         AS day_of_week,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(daily_gross), 2)                  AS avg_daily_earnings,
    ROUND(AVG(daily_eur_per_hour), 2)           AS avg_eur_per_hour
FROM (
    SELECT
        Date,
        SUM(Driver_Earnings_EUR)                AS daily_gross,
        SUM(Driver_Earnings_EUR) /
            (SUM(Duration_Min) / 60.0)          AS daily_eur_per_hour
    FROM rides
    GROUP BY Date
) daily
JOIN rides r USING (Date)
GROUP BY day_of_week
ORDER BY day_of_week;
"""
print("Q3.1 — Average Earnings by Day of Week")
print("-" * 65)
print(run_query(q3_1, conn).to_string(index=False))

# Q3.2 — Best hours of the day
q3_2 = """
SELECT
    CAST(strftime('%H', Start_Time) AS INTEGER) AS hour,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour,
    ROUND(SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                   AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                   THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 1)                      AS acceptance_rate_pct
FROM rides
WHERE CAST(strftime('%H', Start_Time) AS INTEGER) >= 9
GROUP BY hour
HAVING total_rides >= 10
ORDER BY hour;
"""
print("\nQ3.2 — Performance by Hour of Day")
print("-" * 65)
print(run_query(q3_2, conn).to_string(index=False))

# Q3.3 — Monthly progression
q3_3 = """
SELECT
    strftime('%Y-%m', Date)                     AS month,
    COUNT(DISTINCT Date)                        AS days_worked,
    COUNT(*)                                    AS total_rides,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour,
    ROUND(SUM(Driver_Earnings_EUR) /
              COUNT(DISTINCT Date), 2)          AS avg_daily_earnings
FROM rides
GROUP BY month
ORDER BY month;
"""
print("\nQ3.3 — Monthly Progression")
print("-" * 65)
print(run_query(q3_3, conn).to_string(index=False))


Q3.1 — Average Earnings by Day of Week
-----------------------------------------------------------------
day_of_week  days_worked  total_rides  avg_daily_earnings  avg_eur_per_hour
   1-Monday           14          237              106.34             22.30
  2-Tuesday           13          212              102.95             23.06
3-Wednesday           10          164              113.31             22.41
 4-Thursday           15          244              101.19             22.43
   5-Friday           15          255              118.40             24.20
 6-Saturday           14          214              105.24             23.95
   7-Sunday           15          219              104.70             24.32

Q3.2 — Performance by Hour of Day
-----------------------------------------------------------------
 hour  total_rides  avg_earnings  avg_eur_per_km  avg_eur_per_hour  acceptance_rate_pct
    9           13          7.21            0.88             27.78                 84.6
   11     

In [5]:
# ── GROUP 4: ACCEPTANCE CRITERIA & PROFITABILITY ────────────────────

# Q4.1 — Acceptance criteria compliance overall
q4_1 = """
SELECT
    criteria_status,
    COUNT(*)                                    AS total_rides,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rides), 1)
                                                AS pct_rides,
    ROUND(AVG(Driver_Earnings_EUR), 2)          AS avg_earnings,
    ROUND(AVG(Driver_Earnings_EUR /
              Distance_Km), 2)                  AS avg_eur_per_km,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM (
    SELECT *,
        CASE
            WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
             AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                THEN 'Meets both criteria'
            WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                THEN 'Only EUR/km met'
            WHEN Driver_Earnings_EUR / Duration_Min * 60 >= 12
                THEN 'Only EUR/hour met'
            ELSE 'Neither met'
        END AS criteria_status
    FROM rides
)
GROUP BY criteria_status
ORDER BY total_rides DESC;
"""
print("Q4.1 — Acceptance Criteria Compliance")
print("-" * 75)
print(run_query(q4_1, conn).to_string(index=False))

# Q4.2 — Acceptance rate by platform
q4_2 = """
SELECT
    Platform,
    COUNT(*)                                    AS total_rides,
    SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
              AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
             THEN 1 ELSE 0 END)                 AS meets_both,
    ROUND(SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                    AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                   THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 1)                      AS acceptance_rate_pct,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)            AS avg_eur_per_hour
FROM rides
GROUP BY Platform
ORDER BY acceptance_rate_pct DESC;
"""
print("\nQ4.2 — Acceptance Rate by Platform")
print("-" * 75)
print(run_query(q4_2, conn).to_string(index=False))

# Q4.3 — Top 10 most profitable individual rides
q4_3 = """
SELECT
    Ride_ID,
    Platform,
    Category,
    Date,
    ROUND(Distance_Km, 2)                       AS km,
    ROUND(Duration_Min, 0)                      AS duration_min,
    ROUND(Driver_Earnings_EUR, 2)               AS earnings,
    ROUND(Driver_Earnings_EUR / Distance_Km, 2) AS eur_per_km,
    ROUND(Driver_Earnings_EUR /
          Duration_Min * 60, 2)                 AS eur_per_hour
FROM rides
ORDER BY eur_per_hour DESC
LIMIT 10;
"""
print("\nQ4.3 — Top 10 Most Efficient Rides (by €/hour)")
print("-" * 75)
print(run_query(q4_3, conn).to_string(index=False))

# Q4.4 — Net profit summary with fuel cost
q4_4 = """
SELECT
    COUNT(DISTINCT Date)                        AS days_worked,
    ROUND(SUM(Driver_Earnings_EUR), 2)          AS total_gross,
    ROUND(SUM(Distance_Km) * 0.0976, 2)        AS total_fuel_cost,
    CAST(ROUND((julianday(MAX(Date)) -
                julianday(MIN(Date))) / 7)
         AS INTEGER) * 30                       AS total_fleet_cost,
    ROUND(SUM(Driver_Earnings_EUR) -
          SUM(Distance_Km) * 0.0976, 2)        AS net_before_fleet,
    ROUND(SUM(Driver_Earnings_EUR) -
          SUM(Distance_Km) * 0.0976 -
          (CAST(ROUND((julianday(MAX(Date)) -
                       julianday(MIN(Date))) / 7)
           AS INTEGER) * 30), 2)               AS net_after_all_costs,
    ROUND(AVG(Driver_Earnings_EUR /
              Duration_Min * 60), 2)           AS avg_eur_per_hour
FROM rides;
"""
print("\nQ4.4 — Net Profit Summary")
print("-" * 75)
print(run_query(q4_4, conn).to_string(index=False))

Q4.1 — Acceptance Criteria Compliance
---------------------------------------------------------------------------
    criteria_status  total_rides  pct_rides  avg_earnings  avg_eur_per_km  avg_eur_per_hour
Meets both criteria         1269       82.1          6.14            0.92             25.80
  Only EUR/hour met          272       17.6          7.88            0.44             21.91
    Only EUR/km met            3        0.2          6.69            0.62             10.48
        Neither met            1        0.1          4.69            0.48             10.82

Q4.2 — Acceptance Rate by Platform
---------------------------------------------------------------------------
Platform  total_rides  meets_both  acceptance_rate_pct  avg_eur_per_hour
    Bolt          751         636                 84.7             24.64
    Uber          794         633                 79.7             25.48

Q4.3 — Top 10 Most Efficient Rides (by €/hour)
-----------------------------------------------

In [6]:
# ── FINAL SUMMARY ───────────────────────────────────────────────────

summary = """
SELECT
    'Total Rides'           AS metric,
    CAST(COUNT(*) AS TEXT)  AS value
FROM rides
UNION ALL
SELECT 'Total Gross (EUR)',
    CAST(ROUND(SUM(Driver_Earnings_EUR), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Net Profit after All Costs (EUR)',
    CAST(ROUND(SUM(Driver_Earnings_EUR) -
         SUM(Distance_Km) * 0.0976 -
         (CAST(ROUND((julianday(MAX(Date)) -
                      julianday(MIN(Date))) / 7)
          AS INTEGER) * 30), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Avg EUR/hour',
    CAST(ROUND(AVG(Driver_Earnings_EUR /
         Duration_Min * 60), 2) AS TEXT)
FROM rides
UNION ALL
SELECT 'Best Platform (acceptance rate)',
    Platform
FROM (
    SELECT Platform,
           SUM(CASE WHEN Driver_Earnings_EUR / Distance_Km >= 0.50
                     AND Driver_Earnings_EUR / Duration_Min * 60 >= 12
                    THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS rate
    FROM rides GROUP BY Platform ORDER BY rate DESC LIMIT 1
)
UNION ALL
SELECT 'Best Day of Week',
    day_of_week
FROM (
    SELECT
        CASE CAST(strftime('%w', Date) AS INTEGER)
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_of_week,
        AVG(daily_gross) AS avg_gross
    FROM (
        SELECT Date,
               SUM(Driver_Earnings_EUR) AS daily_gross
        FROM rides GROUP BY Date
    ) d
    JOIN rides r USING(Date)
    GROUP BY day_of_week
    ORDER BY avg_gross DESC
    LIMIT 1
)
UNION ALL
SELECT 'Golden Window', '16:00 - 18:00 (91-93% acceptance rate)';
"""

print("=" * 75)
print("SQL ANALYSIS — FINAL SUMMARY")
print("=" * 75)
print(run_query(summary, conn).to_string(index=False))
print("=" * 75)

SQL ANALYSIS — FINAL SUMMARY
                          metric                                  value
                     Total Rides                                   1545
               Total Gross (EUR)                                9961.36
Net Profit after All Costs (EUR)                                7996.48
                    Avg EUR/hour                                  25.07
 Best Platform (acceptance rate)                                   Bolt
                Best Day of Week                                 Friday
                   Golden Window 16:00 - 18:00 (91-93% acceptance rate)


# SQL Analysis — TVDE Lisbon Dataset

## Overview
This notebook replicates and extends the Python analysis using SQL queries
via SQLite, demonstrating proficiency in both tools for data analysis.

## Database Setup
- **Source:** tvde_rides_clean.csv (1,512 rides, cleaned in notebook 01)
- **Engine:** SQLite (in-memory via Python's sqlite3 library)
- **Table:** rides (12 columns, 1,512 rows)

## Queries Performed

### Group 1 — Overview & Volume
- Dataset overview (total rides, km, hours, earnings)
- Rides and earnings by platform
- Rides and efficiency by category

### Group 2 — Earnings & Profitability
- Top 10 best earning days
- Daily performance vs 100€ target
- Earnings efficiency by distance bracket

### Group 3 — Temporal Patterns
- Average earnings by day of week
- Performance and acceptance rate by hour of day
- Monthly progression of earnings

### Group 4 — Acceptance Criteria & Profitability
- Acceptance criteria compliance breakdown
- Acceptance rate by platform
- Top 10 most efficient rides by €/hour
- Net profit summary after all operational costs

## Key SQL Findings
| Metric | Value |
|---|---|
| Total rides | 1,512 |
| Total gross earnings | 9,744 € |
| Total fuel cost | 1,453 € |
| Total fleet cost | 450 € (15 weeks × 30 €) |
| Net profit after all costs | 7,841 € |
| Average €/hour | 25.01 € |
| Best platform (acceptance rate) | Bolt (84.7%) |
| Best day of week | Friday (118 € avg) |
| Golden window | 16:00–18:00 (91–93% acceptance) |

## Cross-Validation
Net profit was validated against Power BI and Python — all three
tools return **7,841.25 €**, confirming consistency across the
full analysis pipeline.

## Tools Used
- Python `sqlite3` — database creation and query execution
- `pandas.read_sql_query()` — result retrieval and display
- SQL features used: `CASE WHEN`, `GROUP BY`, `HAVING`,
  `subqueries`, `UNION ALL`, `strftime()`, `julianday()`,
  `ROUND()`, `CAST()`